In [45]:
from Montreal_UHI_toolbox import *
from branca.element import Element
from folium.elements import MacroElement
from jinja2 import Template
import re

In [46]:
class FloatImageWithID(MacroElement):
    _template = Template("""
        {% macro script(this, kwargs) %}
        var img = L.DomUtil.create('img', '');
        img.setAttribute('src', '{{ this.image }}');
        img.style.width = '{{ this.width }}';
        img.style.height = '{{ this.height }}';

        var div = L.DomUtil.create('div', 'leaflet-control float-image');
        div.id = '{{ this.div_id }}';
        div.style.display = 'none';  // hide on load
        div.appendChild(img);

        L.Control.FloatImage{{ this.div_id | replace('-', '_') }} = L.Control.extend({
            onAdd: function(map) {
                return div;
            },
            onRemove: function(map) {}
        });

        L.control.floatImage{{ this.div_id | replace('-', '_') }} = function(opts) {
            return new L.Control.FloatImage{{ this.div_id | replace('-', '_') }}(opts);
        };

        L.control.floatImage{{ this.div_id | replace('-', '_') }}({ position: '{{ this.position }}' }).addTo({{ this._parent.get_name() }});
        {% endmacro %}
    """)

    def __init__(self, image, position='topleft', width='auto', height='auto', div_id='float-image'):
        super().__init__()
        self._name = 'FloatImage'
        self.image = image
        self.position = position
        self.width = width
        self.height = height
        self.div_id = div_id




class ColorbarToggleScript(MacroElement):
    def __init__(self, field_names):
        super().__init__()
        self._name = 'ColorbarToggleScript'
        self._template = Template(f"""
            {{% macro script(this, kwargs) %}}
            function findMap(retries = 10, delay = 300) {{
                return new Promise((resolve, reject) => {{
                    let attempts = 0;
                    function tryFind() {{
                        const map = Object.values(window).find(obj => obj instanceof L.Map);
                        if (map) {{
                            resolve(map);
                        }} else if (++attempts < retries) {{
                            setTimeout(tryFind, delay);
                        }} else {{
                            console.error("Map object not found after retries.");
                            reject("Map not found");
                        }}
                    }}
                    tryFind();
                }});
            }}

            findMap().then((map) => {{
                console.log("Map found!", map);

                const colorbarMap = {{
                    {', '.join([f'"{name}": "colorbar-{name}"' for name in field_names])}
                }};

                function show(id) {{
                    const el = document.getElementById(id);
                    if (el) {{
                        el.style.display = "block";
                        console.log("SHOWING", id);
                    }}
                }}

                function hide(id) {{
                    const el = document.getElementById(id);
                    if (el) {{
                        el.style.display = "none";
                        console.log("HIDING", id);
                    }}
                }}

                Object.values(colorbarMap).forEach(hide);

                map.on('overlayadd', (e) => {{
                    console.log("overlayadd:", e.name);
                    const id = colorbarMap[e.name];
                    if (id) show(id);
                }});

                map.on('overlayremove', (e) => {{
                    console.log("overlayremove:", e.name);
                    const id = colorbarMap[e.name];
                    if (id) hide(id);
                }});

                document.querySelectorAll(
                    ".leaflet-control-layers-group input[type='checkbox'], \
                     .leaflet-control-layers-group input[type='radio']"
                ).forEach((input) => {{
                    input.addEventListener('change', function () {{
                        const label = this.closest("label");
                        if (!label) return;

                        const name = label.innerText.trim();
                        const eventType = this.checked ? "overlayadd" : "overlayremove";
                        map.fire(eventType, {{ name: name }});
                    }});
                }});
            }});
            {{% endmacro %}}
        """)


In [47]:
def draw_map_layers(fields=[],cmap_name_array=[],vmins=[],vmaxs=[],num_level_array=[],stations_visible=True,opacity=0.7,cmocean_CMAP_array=[]):
    """
    Creates a map and displays the station_locations within the bounds of the simulation on it. 
    It then generates an ImageOverlay of some the static field generated.
    Example usage:
        > m = draw_map_layers([static_fields_C['orog'],static_fields_C['urban_frac']])
        > display(m)

    parameters:
        fields - xarray.core.dataarray.DataArray
            2D temperature, humidity, or other data 
        cmaps - string
            matplotlib colormap indicator
        vmin, vmax - float 
            minimum and maximum 
    returns:
        m - folium.folium.Map
    """
    # Ensure other parameters are the same length as fields
    cmap_name_array = pad_list(cmap_name_array,len(fields),'bwr')
    vmins = pad_list(vmins,len(fields),None)
    vmaxs = pad_list(vmaxs,len(fields),None)
    num_level_array = pad_list(num_level_array,len(fields),None)
    cmocean_CMAP_array = pad_list(cmocean_CMAP_array,len(fields),None)
    
    # Initialize the map
    m = folium.Map(location=[centre_lat, centre_lon], zoom_start=8)

    folium.TileLayer('cartodb positron').add_to(m)

    if stations_visible:
        # Add station markers
        folium.GeoJson(
            geojson_stations,
            popup=folium.GeoJsonPopup(fields=['station_name'], aliases=['Station Name']),
            marker=folium.CircleMarker(radius=3, color='grey', fill=True, fill_color='grey', fill_opacity=opacity),
            show=False,
            name='stations'
        ).add_to(m)

    # Overlay field data
    overlays = []
    field_names = []
    for field,vmin,vmax,cmap_name,num_levels,cmocean_CMAP in zip(fields,vmins,vmaxs,cmap_name_array,num_level_array,cmocean_CMAP_array):
        if field is not None:
            # Create an image to store the field data
            image_buffer = io.BytesIO()
            field_ravel = np.ravel(field.values)
            field.name = re.sub(r'\W|^(?=\d)', '_', field.name)
            field_names.append(field.name)
            if vmin == None:
                vmin = min(field_ravel)
            if vmax == None:
                vmax = max(field_ravel)
            
            # Colorbar properties
            norm = Normalize(vmin=vmin, vmax=vmax)
            cmap = plt.get_cmap(cmap_name)
            sm = ScalarMappable(cmap=cmap, norm=norm)
            
            # Create the figure to save to the map
            fig, ax = plt.subplots(figsize=(8, 8))
            collection = PatchCollection(
                patches,
                linewidth=0,
                edgecolor='none',
                antialiased=False
            )
            collection.set_array(field_ravel)
            collection.set_cmap(cmap)
            collection.set_clim(vmin, vmax)
            
            ax.add_collection(collection)
            ax.autoscale_view()
            ax.set_aspect('equal')
            ax.axis('off')
    
            minx, miny, maxx, maxy = gdf.total_bounds
            ax.set_xlim(minx, maxx)
            ax.set_ylim(miny, maxy)    
            ax.axis('off')
            # Save image to buffer
            plt.savefig(image_buffer, format='png', bbox_inches='tight', pad_inches=0, transparent=True)
            plt.close(fig)
            plt.close()
            image_buffer.seek(0)
            image_base64 = base64.b64encode(image_buffer.read()).decode()
            image_uri = f"data:image/png;base64,{image_base64}"
            
            overlays.append(ImageOverlay(
                image=image_uri,
                bounds=bounds,
                opacity=opacity,
                interactive=True,
                cross_origin=False,
                pixelated=True,
                show=False,
                name=field.name,
                group='Field'
            ))
            overlays[-1].add_to(m)
            
            # Add the colourbar
            fig, ax = plt.subplots(figsize=(2, 0.2))  # Width x Height in inches

            # Create the colorbar from ScalarMappable
            cb = fig.colorbar(sm, cax=ax,orientation='horizontal')
            cb.set_ticks([vmin,vmax])
            cb.set_label(f'{field.name} ({field.attrs['units']})')
            
            # Save colorbar to buffer
            image_buffer_cbar = io.BytesIO()
            plt.savefig(image_buffer_cbar, format='png',bbox_inches='tight', pad_inches=0, transparent=True)
            plt.close(fig)
            image_buffer_cbar.seek(0)
            image_base64 = base64.b64encode(image_buffer_cbar.read()).decode()
            image_uri_cbar = f"data:image/png;base64,{image_base64}"

            # FloatImage(image_uri_cbar, bottom=90+adjB, left=5).add_to(m)
            FloatImageWithID(
                image=image_uri_cbar,
                div_id=f"colorbar-{field.name}"
            ).add_to(m)

           
    
    folium.LayerControl(collapsed=False).add_to(m)
    overlays.append(
        folium.TileLayer(
        tiles='',
        name='Clear',
        overlay=True,
        attr='none',
        control=True
    ))
    overlays[-1].add_to(m)
    
    GroupedLayerControl(groups={
                'Fields': overlays,
            }, collapsed=False).add_to(m)
    m.fit_bounds(bounds)
    m.get_root().add_child(ColorbarToggleScript(field_names))
    return m

In [ ]:
m = draw_map_layers([static_fields_C['orog'],static_fields_C['urban_frac']])
# m.save('/runoff/gulley/test_map.html')
display(m)



In [ ]:
from Montreal_UHI_toolbox import static_fields_C, draw_map_layers
m = draw_map_layers([static_fields_C['orog'],static_fields_C['urban_frac']])
display(m)